# 04. Modélisation Machine Learning, Indicateurs Techniques & Dérivés (`^GVZ`)

**Projet :** Prévision de l'or (XAU/USD / `GC=F`) — *Advanced Time Series* (ENSAI, 3A)

### Objectifs :
1. Construire des variables techniques stationnarisées (**RSI 14j**, **MACD normalisé**, **Range/ATR**, **volatilité réalisée**).
2. Intégrer les covariables macroéconomiques (**DXY**, **VIX**, **TNX_10Y**, **OIL**, **SP500**).
3. Tester l'apport spécifique des **données de dérivés sur l'or** : le **CBOE Gold ETF Volatility Index (`GVZ`)** (volatilité implicite des options sur l'or, *forward-looking*).
4. Comparer **avec vs sans dérivés (`GVZ`)** sur les rendements ($r_t$) et sur la volatilité ($|r_t|$) face à **Naïf** et **GARCH(1,1)-t** sur la période hors échantillon (252 derniers jours = 1 an de bourse).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from arch import arch_model

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Feature Engineering (Indicateurs techniques + Macro + Dérivés `GVZ`)

In [ ]:
df_gold = pd.read_csv(DATA_DIR / "gold_daily.csv", parse_dates=["Date"]).set_index("Date")
df_cov = pd.read_csv(DATA_DIR / "covariates_daily.csv", parse_dates=["Date"]).set_index("Date")
df = df_gold.join(df_cov, how="left").ffill()

# Cibles
df["return"] = np.log(df["Close"]).diff()
df["abs_return"] = np.abs(df["return"])

# Indicateurs techniques stationnaires
delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df["RSI_14"] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD_norm"] = (ema12 - ema26) / ema26
df["MACD_hist_norm"] = df["MACD_norm"] - df["MACD_norm"].ewm(span=9, adjust=False).mean()

df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
df["vol_5d"] = df["return"].rolling(5).std()
df["vol_21d"] = df["return"].rolling(21).std()

# Macro & Dérivés
df["DXY_ret"] = np.log(df["DXY"]).diff()
df["TNX_diff"] = df["TNX_10Y"].diff()
df["OIL_ret"] = np.log(df["OIL"].clip(lower=0.01)).diff()
df["SP500_ret"] = np.log(df["SP500"]).diff()
df["GVZ_diff"] = df["GVZ"].diff()
df["GVZ_ratio_VIX"] = df["GVZ"] / df["VIX"].replace(0, np.nan)

# Décalage strict t-1 (zéro fuite temporelle)
base_cols = [
    "return", "abs_return", "RSI_14", "MACD_norm", "MACD_hist_norm",
    "range_pct", "vol_5d", "vol_21d", "DXY_ret", "VIX", "TNX_diff", "OIL_ret", "SP500_ret"
]
deriv_cols = ["GVZ", "GVZ_diff", "GVZ_ratio_VIX"]

X = pd.DataFrame(index=df.index)
for c in base_cols + deriv_cols:
    X[f"{c}_lag1"] = df[c].shift(1)
for k in [2, 3, 5]:
    X[f"return_lag{k}"] = df["return"].shift(k)
    X[f"abs_return_lag{k}"] = df["abs_return"].shift(k)

data = X.join(df[["return", "abs_return"]]).dropna()
feats_no_deriv = [c for c in X.columns if not c.startswith("GVZ")]
feats_with_deriv = list(X.columns)

N_TEST = 252  # 1 an de bourse hors échantillon
train = data.iloc[:-N_TEST]
test = data.iloc[-N_TEST:]
print(f"Données ML prêtes (avec ^GVZ depuis {data.index.min().date()}) : Train={len(train)} | Test={len(test)}")

## 2. Prévision des **rendements journaliers ($r_t$)** : Impact des dérivés

In [ ]:
y_tr, y_te = train["return"].values, test["return"].values

ridge_no = make_pipeline(StandardScaler(), Ridge(alpha=100.0)).fit(train[feats_no_deriv], y_tr)
ridge_gvz = make_pipeline(StandardScaler(), Ridge(alpha=100.0)).fit(train[feats_with_deriv], y_tr)

gb_no = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42).fit(train[feats_no_deriv], y_tr)
gb_gvz = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42).fit(train[feats_with_deriv], y_tr)

def eval_ret(name, yp):
    return {
        "Modèle": name,
        "MAE": mean_absolute_error(y_te, yp),
        "RMSE": root_mean_squared_error(y_te, yp),
        "Hit Ratio": np.mean(np.sign(y_te) == np.sign(yp)) if np.any(yp != 0) else 0.50
    }

res_ret = pd.DataFrame([
    eval_ret("Naïf (r_t = 0)", np.zeros_like(y_te)),
    eval_ret("Ridge (Tech + Macro)", ridge_no.predict(test[feats_no_deriv])),
    eval_ret("Ridge (+ Dérivés GVZ)", ridge_gvz.predict(test[feats_with_deriv])),
    eval_ret("Boosting (Tech + Macro)", gb_no.predict(test[feats_no_deriv])),
    eval_ret("Boosting (+ Dérivés GVZ)", gb_gvz.predict(test[feats_with_deriv])),
])
print("=== RENDEMENTS JOURNALIERS (252j test) ===")
print(res_ret.to_string(index=False))

## 3. Prévision de la **volatilité journalière ($|r_t|$)** : GARCH vs ML vs Dérivés `^GVZ`

In [ ]:
v_tr, v_te = train["abs_return"].values, test["abs_return"].values
v_naive = test["abs_return_lag1"].values

# GARCH(1,1) Student-t
scale = 100.0
am = arch_model(data["return"] * scale, vol="Garch", p=1, q=1, dist="t")
garch_res = am.fit(last_obs=len(train), disp="off")
garch_fc = garch_res.forecast(start=len(train), horizon=1, reindex=False)
pred_garch = (np.sqrt(garch_fc.variance.iloc[-N_TEST:, 0].values) / scale) * np.sqrt(2 / np.pi)

# ML sans vs avec GVZ
gb_v_no = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42).fit(train[feats_no_deriv], v_tr)
gb_v_gvz = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42).fit(train[feats_with_deriv], v_tr)

pred_v_no = np.clip(gb_v_no.predict(test[feats_no_deriv]), 0, None)
pred_v_gvz = np.clip(gb_v_gvz.predict(test[feats_with_deriv]), 0, None)

res_vol = pd.DataFrame([
    {"Modèle": "Naïf (|r_{t-1}|)", "MAE": mean_absolute_error(v_te, v_naive), "RMSE": root_mean_squared_error(v_te, v_naive)},
    {"Modèle": "GARCH(1,1)-t", "MAE": mean_absolute_error(v_te, pred_garch), "RMSE": root_mean_squared_error(v_te, pred_garch)},
    {"Modèle": "Boosting (Tech + Macro)", "MAE": mean_absolute_error(v_te, pred_v_no), "RMSE": root_mean_squared_error(v_te, pred_v_no)},
    {"Modèle": "Boosting (+ Dérivés GVZ)", "MAE": mean_absolute_error(v_te, pred_v_gvz), "RMSE": root_mean_squared_error(v_te, pred_v_gvz)},
])
print("\n=== VOLATILITÉ JOURNALIÈRE |r_t| (252j test) ===")
print(res_vol.to_string(index=False))

plt.figure(figsize=(13, 4))
plt.plot(test.index, v_te, label="Réalisé |r_t|", color="black", alpha=0.45)
plt.plot(test.index, pred_garch, label="GARCH(1,1)-t", color="#1f77b4", lw=1.5)
plt.plot(test.index, pred_v_gvz, label="Boosting (+ Dérivés ^GVZ)", color="#d62728", lw=1.5)
plt.title("Volatilité de l'or (252j test) : GARCH(1,1)-t vs Boosting + Options ^GVZ", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "04_ml_gvz_volatility.png", dpi=300)
plt.show()